[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

The least-squares loss is

$$
L(w,b) = \|Xw + b - y\|_2^2
$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$W_{\text{aug}} =\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

where

$$
X_{\text{aug}} = \begin{bmatrix} X & \mathbf{1} \end{bmatrix}
$$

and

$$
\theta =
\begin{bmatrix}
w \\
b
\end{bmatrix}
$$

Then

$$
X_{\text{aug}}\theta
=
\begin{bmatrix} X & \mathbf{1} \end{bmatrix}
\begin{bmatrix}
w \\
b
\end{bmatrix}
$$

Therefore,

$$
X_{\text{aug}}\theta = Xw + b
$$

so 
$$
X_{\text{aug}} = X
$$ 
if no bias

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [ ]:

# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        X_aug = torch.cat([X, torch.ones(X.shape[0], 1)], dim=1)  # Add bias term
        print(X_aug.shape)
        w_aug = torch.inverse(X_aug.T @ X_aug) @ X_aug.T @ y
        w = w_aug[:-1]  # Extract weights
        b = w_aug[-1]   # Extract bias
        return w, b
        pass  # Return (w, b)

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""
        w, b = torch.zeros(X.shape[1]), torch.zeros(1)  # Initialize parameters
        X_pred = X @ w + b  # Initial predictions
        for _ in range(steps):
            error = X_pred - y
            grad_w = (2 / X.shape[0]) * X.T @ error # dL/dw
            grad_b = (2 / X.shape[0]) * error.sum() # dL/db
            w -= lr * grad_w
            b -= lr * grad_b
            X_pred = X @ w + b  # Update predictions
        return w, b
        pass  # Return (w, b)

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        weight = nn.Linear(X.shape[1], 1)  # Create linear layer
        criterion = nn.MSELoss()  # Mean Squared Error loss
        optimizer = torch.optim.SGD(weight.parameters(), lr=lr)  # SGD optimizer
        for _ in range(steps):
            outputs = weight(X).squeeze()  # Forward pass
            loss = criterion(outputs, y)
            loss.backward()  # Backward pass
            optimizer.step()  # Update parameters
            optimizer.zero_grad()  # Zero gradients
        return weight.weight.data.squeeze(), weight.bias.data.squeeze()
        pass  # Return (w, b)

In [16]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

torch.Size([100, 4])
Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [17]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
torch.Size([50, 4])
  ✅ [1/6] Closed-form returns correct shapes (31.7ms)
torch.Size([100, 4])
  ✅ [2/6] Closed-form finds correct weights (0.6ms)
  ✅ [3/6] Gradient descent converges (28.2ms)
  ✅ [4/6] nn.Linear approach works (102.1ms)
torch.Size([200, 3])
  ✅ [5/6] All three methods agree (171.8ms)
torch.Size([30, 3])
  ✅ [6/6] Closed-form uses no autograd (0.2ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (334.5ms total)
  Progress saved. Run status() to see your dashboard.

